# The LVGP with the recycle cycle as an input (2026-11)

**Why.** `lvgp_vs_mixed.ipynb` fits 12 cycle-0 means with PAL % and temperature, so it cannot see recycling. Here the modeling unit is the **run** (a formulation at one cycle, 24 run means with the 4 Nov TPU data) and the cycle is a third continuous input, so cycle 2 sits between 1 and 3. The polymer stays categorical, through the latent map. LVGP chosen in `lvgp_vs_mixed.ipynb`.

$$\ln y(\mathbf{x},c)\sim\mathcal{GP}(\mu,k),\quad \mathbf{x}=(\mathrm{PAL},T,\mathrm{cycle}),\qquad k=\sigma^2\exp\Big(-\tfrac12\sum_j \frac{(x_j-x_j')^2}{\ell_j^2}\Big)\exp\big(-\tfrac12\lVert \mathbf{z}(c)-\mathbf{z}(c')\rVert^2\big)$$

$$\bar y_{\mathrm{run}}=\ln y(\mathbf{x},c)+\epsilon,\qquad \epsilon\sim\mathcal{N}\big(0,\ s^2_{\mathrm{run}}/n_{\mathrm{run}}\big)$$

**Key points**
* Knowing the cycle predicts a left-out recycled run better on all four responses: RMSE of log(response) 0.26 vs 0.31 (UTS), 0.23 vs 0.25 (modulus), 0.30 vs 0.35 (strain), 0.41 vs 0.80 (toughness).
* Average gain over the PAL–temperature range: UTS +20 % PP, +23 % HDPE; modulus +9, +12; strain +15, +21; toughness +36, +73.
* **Its intervals are too narrow:** the noise counts replicates, not batch differences; only 38–88 % of left-out recycled runs fall within ±2 SD. Use it for shape and direction, and the paired test in `notebooks/data_processing_v5.ipynb` for whether a change is real. Next step: learn an extra batch noise term.
* TPU has no recycled data: it is not drawn at cycle 1.
* lvgp_mixed with the cycle input (section 6): a split decision against the LVGP; same conclusions under both.
* Warping: PAL flat, a hint of a valley near 210 °C, but it does not yet beat the polymer average on a left-out formulation (0.44 vs 0.38).

Figures are written only if a `SAVE_FIG` switch is on.

In [ ]:
import os, sys, warnings
from pathlib import Path

HERE = Path.cwd()
MODELING = HERE if (HERE / "lvgp_compare.py").exists() else HERE / "modeling"
ROOT = MODELING.parent
for p_ in (str(MODELING), str(ROOT)):
    if p_ not in sys.path:
        sys.path.insert(0, p_)
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

import lvgp_compare as LC
from models import MODELS
from utils import analysis as A, plots

torch.set_num_threads(4)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
FIG_DIR = MODELING / "figures"                   # used only when a SAVE_FIG switch is True
DELIVERY = "11_04_26"
kept = LC.load_tensile(str(ROOT), delivery=DELIVERY)
print(f"{len(kept)} specimens, {kept.groupby(['formulation', 'recycled_x']).ngroups} runs")
LC.run_table(kept, "max_tensile_stress_MPa")[["formulation", "cycle", "polymer", "pal", "temp", "n"]]

---
## 1 — Fit, and what the model learned

Length scale per input (scaled range 0–1; one loop = 0.33 on the cycle axis): small = changes fast. Latent map: close polymers share information.

In [ ]:
fits = {r: LC.fit_cycle(kept, r) for r in LC.RESPONSES}
tables = {r: LC.run_table(kept, r) for r in LC.RESPONSES}
rows = []
for r, m in fits.items():
    z = LC.latent_positions(m)
    d = lambda a, b: float(np.linalg.norm(z.loc[a].to_numpy() - z.loc[b].to_numpy()))
    rows.append({"response": LC.RESPONSES[r], **LC.length_scales(m), "PP-HDPE": d("PP+PAL", "HDPE+PAL"), "PP-TPU": d("PP+PAL", "TPU+PAL")})
pd.DataFrame(rows).round(2)

In [ ]:
pd.concat({LC.RESPONSES[r]: LC.average_ratio(m) for r, m in fits.items()}).round(3)   # average gain over the range, 5-95 %

---
## 2 — Surfaces

PAL × temperature at cycle 0, at cycle 1, and their ratio. Trust them near the markers; corners are extrapolations. Bands exclude 1 too readily (see key points).

In [ ]:
SAVE_FIG = False                                 # True -> write the figures to modeling/figures/
surfs = {r: LC.cycle_surfaces(m) for r, m in fits.items()}
for r in LC.RESPONSES:
    save = None
    if SAVE_FIG:
        FIG_DIR.mkdir(exist_ok=True); save = FIG_DIR / f"surface_{r}.png"
    display(plots.lvgp_surfaces(surfs[r], tables[r], LC.RESPONSES[r], save=save))

In [ ]:
for r in LC.RESPONSES:                           # the same surfaces in 3-D
    display(plots.lvgp_surface_3d(surfs[r], tables[r], LC.RESPONSES[r]))

---
## 3 — Partial dependence by cycle

In [ ]:
have = set(zip(kept["polymer_system"], kept["recycled_x"].astype(int)))
pds = {r: LC.partial_dependence_cycle(m, have=have) for r, m in fits.items()}
plots.partial_dependence_cycles(pds)

---
## 4 — Does the cycle input help? Leave one run out

Each run is left out in turn, its partner at the other cycle stays in. RMSE of log(response), and the share within ±2 SD (95 % if calibrated). About 3 min.

In [ ]:
with_c = pd.concat([LC.leave_one_run_out(kept, r, workers=12) for r in LC.RESPONSES], ignore_index=True)
no_c = pd.concat([LC.leave_one_run_out(kept, r, workers=12, use_cycle=False) for r in LC.RESPONSES], ignore_index=True)
score = lambda d: pd.Series({"rmse": float(np.sqrt((d["err"] ** 2).mean())), "within_2sd": float((d["z"].abs() <= 2).mean())})
pd.concat({"with cycle": with_c[with_c["cycle"] == 1].groupby("response").apply(score),
           "without": no_c[no_c["cycle"] == 1].groupby("response").apply(score)}, axis=1).round(3)

---
## 5 — Printability: warping as a response

Same model on the log of the mean corner warping per formulation (11 means; F09 missing, cycle not stated). If leaving one formulation out does not beat the polymer's average, PAL and temperature are not yet shown to help.

In [ ]:
design = pd.read_csv(ROOT / "doe" / "doe_formulations.csv")
warp = A.load_warping(ROOT / "raw_data" / DELIVERY / "extracted" / "printability" / "Sample Warping.xlsx", design)
torch.manual_seed(0)
wm = MODELS["lvgp"].fit(LC.warping_data(warp), needs_r=False, bounds=LC.BOUNDS)
pal, temp = np.linspace(*LC.BOUNDS[0], 41), np.linspace(*LC.BOUNDS[1], 41)
P, T = np.meshgrid(pal, temp, indexing="ij")
ws = dict(pal=pal, temp=temp, surf={p: np.exp(wm.predict(lv, np.column_stack([P.ravel(), T.ravel()]))[0]).reshape(P.shape)
                                    for lv, p in enumerate(LC.POLYMERS, 1)})
plots.warping_surfaces(ws, warp)

In [ ]:
lo = LC.warping_lofo(warp)
print("RMSE of log(warping): LVGP", round(float(np.sqrt((lo["err_lvgp"] ** 2).mean())), 3), "| polymer average", round(float(np.sqrt((lo["err_polymer"] ** 2).mean())), 3))
lo.round(3)

---
## 6 — Which kernel: LVGP or lvgp_mixed

`lvgp_vs_mixed.ipynb` chose the LVGP on the cycle-0 data. Here both are refitted on the **24 run means** (snapshot `11_04_26`) with the cycle input and compared the way a probabilistic model should be:

* **Score: the negative log predictive density (NLPD)** of each left-out run, $-\ln p(\bar y_{\text{out}} \mid \text{model})$ with the predictive variance plus the run's replicate variance. It is a *proper scoring rule*: it rewards a right mean **and** an honest variance, so an over-confident model is punished. RMSE scores the mean only; the share of runs inside $\pm 2$ SD checks calibration. AIC/BIC are not used: in-sample, and the "number of parameters" is ill-defined for a GP with latent positions and MAP hyperparameters. The log marginal likelihood is reported as a side check only (in-sample, prior-sensitive).
* **Paired, fold by fold.** Both kernels see the same folds, so the quantity to look at is the per-fold difference (mean, standard error, how many folds each wins), not two means.
* **Two fold designs.** Leave-one-*run*-out (24 folds) keeps the partner cycle of the same formulation in the training set. Leave-one-*formulation*-out (12 folds, both cycles out) is stricter and mirrors Option 2, predicting a new formulation. Neither tests extrapolation to cycles 2–3.

**Result (leave-one-run-out):** the LVGP is clearly better for UTS (NLPD 1.02 vs 2.39; wins 20 of 24 folds; difference 1.37 ± 0.53); the other three properties are ties (modulus 0.92 vs 1.01, 16 wins; strain 1.24 vs 1.15, 15 wins; toughness 1.83 vs 1.73, 10 wins; all differences within ±1 SE of zero). RMSE mean 0.60 vs 0.65; 69 % vs 64 % within ±2 SD. The mixed kernel's worst fold is F08 at cycle 1 (NLPD 18.8): a confident miss where TPU UTS rose 1.4 → 3.5 MPa. **Leave-one-formulation-out** (12 folds): both kernels do much worse (NLPD UTS 2.67 vs 3.41, modulus 0.34 vs 0.28, strain 2.17 vs 2.10, toughness 3.56 vs 3.04; 59 % vs 65 % within ±2 SD) and tie within ±1 SE except toughness, where lvgp_mixed is a little better (−0.52 ± 0.23). Four formulations per polymer are not enough to predict a fifth; that is a data statement (for Option 2 and for a batch noise term), not a kernel one. **Log evidence** (side check, per run, incl. hyperpriors): LVGP −1.43 / −0.69 / −1.24 / −1.73 vs lvgp_mixed −1.67 / −0.96 / −1.53 / −1.99: higher for the LVGP on all four. **The deck uses the LVGP.** About 6 min.

In [ ]:
mixed = pd.concat([LC.leave_one_run_out(kept, r, workers=12, key="lvgp_mixed") for r in LC.RESPONSES], ignore_index=True)
LC.paired_scores(with_c, mixed).round(2)                     # leave-one-run-out, 24 folds; diff > 0 favours the LVGP

In [ ]:
LC.paired_scores(with_c[with_c["cycle"] == 1], mixed[mixed["cycle"] == 1]).round(2)      # the 12 recycled runs only

**Stricter folds: leave one formulation out** (both cycles held out, 12 folds). This is the test that matches Option 2: can the model predict a formulation it has never seen, at either cycle?

In [ ]:
lofo_lvgp = pd.concat([LC.leave_one_formulation_out(kept, r, workers=12, key="lvgp") for r in LC.RESPONSES], ignore_index=True)
lofo_mixed = pd.concat([LC.leave_one_formulation_out(kept, r, workers=12, key="lvgp_mixed") for r in LC.RESPONSES], ignore_index=True)
LC.paired_scores(lofo_lvgp, lofo_mixed).round(2)

In [ ]:
# side check: log marginal likelihood at the MAP (in-sample, includes the hyperprior terms; higher is better)
pd.DataFrame({LC.RESPONSES[r]: {"LVGP": LC.log_evidence(fits[r]), "lvgp_mixed": LC.log_evidence(LC.fit_cycle(kept, r, key="lvgp_mixed"))} for r in LC.RESPONSES}).T.round(1)

In [ ]:
fits_mixed = {r: LC.fit_cycle(kept, r, key="lvgp_mixed") for r in LC.RESPONSES}
pd.DataFrame({LC.RESPONSES[r]: LC.mixed_shares(m) for r, m in fits_mixed.items()}).T[["share_trend", "share_offset", "share_level"]].round(3)   # how lvgp_mixed splits the variance